# Customer Segmentation Analysis: RFMT & K-Means Clustering


## Executive Summary

This notebook provides a comprehensive customer segmentation analysis using RFMT (Recency, Frequency, Monetary Value, and Tenure) metrics combined with K-Means clustering. We'll explore customer behavior patterns, identify distinct segments, and provide actionable insights for targeted marketing strategies.

**Key Objectives:**
- Perform cohort analysis to understand customer lifecycle patterns
- Calculate RFMT metrics for behavioral segmentation
- Apply K-Means clustering for data-driven customer grouping
- Profile and interpret customer segments for business strategy



## 1. Setup and Data Loading


In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import datetime as dt
from datetime import timedelta

# Visualization libraries
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
import matplotlib.pyplot as plt

# Machine learning libraries
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 2)

import warnings
warnings.filterwarnings('ignore')

print("✓ All libraries loaded successfully")

✓ All libraries loaded successfully


In [2]:
# Load the dataset

online = pd.read_csv('online retail.csv')

print(f"Dataset shape: {online.shape}")
print(f"\nColumns: {list(online.columns)}")
online.head()

Dataset shape: (541909, 8)

Columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


## 2. Data Preprocessing and Cleaning

In [3]:
# Examine data structure
print("Data Info:")
online.info()

print("\nMissing Values:")
print(online.isnull().sum())

print("\nBasic Statistics:")
online.describe()

Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB

Missing Values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Basic Statistics:


,Quantity,UnitPrice,CustomerID
count,541909.00,541909.00,406829.00
mean,9.55,4.61,15287.69
std,218.08,96.76,1713.60
min,-80995.00,-11062.06,12346.00
25%,1.00,1.25,13953.00
50%,3.00,2.08,15152.00
75%,10.00,4.13,16791.00
max,80995.00,38970.00,18287.00


In [4]:
# Data cleaning steps
print("Initial dataset size:", len(online))

# Convert InvoiceDate to datetime
online['InvoiceDate'] = pd.to_datetime(online['InvoiceDate'])

# Remove missing CustomerID (can't segment without customer identifier)
online = online[online['CustomerID'].notna()]
print(f"After removing missing CustomerID: {len(online)}")

# Remove cancelled orders (InvoiceNo starting with 'C')
online = online[~online['InvoiceNo'].astype(str).str.startswith('C')]
print(f"After removing cancellations: {len(online)}")

# Remove negative quantities and prices
online = online[(online['Quantity'] > 0) & (online['UnitPrice'] > 0)]
print(f"After removing negative values: {len(online)}")

# Create TotalSum column
online['TotalSum'] = online['Quantity'] * online['UnitPrice']

# Keep only last 12 months of data for analysis
max_date = online['InvoiceDate'].max()
min_date = max_date - timedelta(days=365)
online = online[online['InvoiceDate'] >= min_date]
print(f"After filtering to last 12 months: {len(online)}")

print(f"\nDate range: {online['InvoiceDate'].min()} to {online['InvoiceDate'].max()}")
print(f"Number of unique customers: {online['CustomerID'].nunique()}")

Initial dataset size: 541909
After removing missing CustomerID: 406829
After removing cancellations: 397924
After removing negative values: 397884
After filtering to last 12 months: 384529

Date range: 2010-12-09 12:59:00 to 2011-12-09 12:50:00
Number of unique customers: 4269


In [5]:
# Create snapshot date for recency calculation (day after last transaction)
snapshot_date = online['InvoiceDate'].max() + timedelta(days=1)
print(f"Analysis snapshot date: {snapshot_date.date()}")

Analysis snapshot date: 2011-12-10


## 3. Cohort Analysis

### 3.1 Building Cohort Structure


In [ ]:
# Helper function to get first day of month
def get_month(x): 
    return dt.datetime(x.year, x.month, 1)

# Assign invoice month
online['InvoiceMonth'] = online['InvoiceDate'].apply(get_month)

# Assign cohort month (first purchase month for each customer)
online['CohortMonth'] = online.groupby('CustomerID')['InvoiceMonth'].transform('min')

print("Sample data with cohort assignment:")
online[['CustomerID', 'InvoiceDate', 'InvoiceMonth', 'CohortMonth']].head(10)

Sample data with cohort assignment:


,CustomerID,InvoiceDate,InvoiceMonth,CohortMonth
20240,14479.0,2010-12-09 12:59:00,2010-12-01,2010-12-01
20241,14479.0,2010-12-09 12:59:00,2010-12-01,2010-12-01
20242,14479.0,2010-12-09 12:59:00,2010-12-01,2010-12-01
20243,14479.0,2010-12-09 12:59:00,2010-12-01,2010-12-01
20244,14479.0,2010-12-09 12:59:00,2010-12-01,2010-12-01
20245,14479.0,2010-12-09 12:59:00,2010-12-01,2010-12-01
20246,14479.0,2010-12-09 12:59:00,2010-12-01,2010-12-01
20247,14479.0,2010-12-09 12:59:00,2010-12-01,2010-12-01
20248,14479.0,2010-12-09 12:59:00,2010-12-01,2010-12-01
20249,14479.0,2010-12-09 12:59:00,2010-12-01,2010-12-01


In [58]:
# Helper function to extract date components
def get_date_int(df, column):
    year = df[column].dt.year
    month = df[column].dt.month
    day = df[column].dt.day
    return year, month, day

# Calculate cohort index (months since first purchase)
invoice_year, invoice_month, _ = get_date_int(online, 'InvoiceMonth')
cohort_year, cohort_month, _ = get_date_int(online, 'CohortMonth')

years_diff = invoice_year.values - cohort_year.values
months_diff = invoice_month.values- cohort_month.values

online['CohortIndex'] = years_diff * 12 + months_diff + 1

print("Cohort Index distribution:")
print(online['CohortIndex'].value_counts().sort_index())

Cohort Index distribution:
CohortIndex
1     114711
2      26081
3      29717
4      25852
5      27202
6      25970
7      24689
8      24235
9      24945
10     20223
11     21684
12      6418
Name: count, dtype: int64


### 3.2 Cohort Metrics Calculation


In [8]:
# Count unique customers per cohort and cohort index
grouping = online.groupby(['CohortMonth', 'CohortIndex'])
cohort_data = grouping['CustomerID'].apply(pd.Series.nunique)
cohort_data = cohort_data.reset_index()

# Create cohort counts pivot table
cohort_counts = cohort_data.pivot(index='CohortMonth', 
                                   columns='CohortIndex',
                                   values='CustomerID')

print("Cohort Counts (Active Customers):")
cohort_counts

Cohort Counts (Active Customers):


CohortIndex,1,2,3,4,5,6,7,8,9,10,11,12,13
CohortMonth,,,,,,,,,,,,,
2010-12-01,509.0,213.0,198.0,226.0,224.0,241.0,216.0,208.0,209.0,225.0,207.0,288.0,164.0
2011-01-01,528.0,141.0,172.0,146.0,193.0,177.0,156.0,155.0,190.0,189.0,217.0,88.0,NaN
2011-02-01,419.0,83.0,83.0,121.0,119.0,105.0,113.0,124.0,111.0,136.0,34.0,NaN,NaN
2011-03-01,493.0,76.0,127.0,106.0,109.0,85.0,139.0,118.0,143.0,46.0,NaN,NaN,NaN
2011-04-01,327.0,75.0,68.0,75.0,70.0,76.0,77.0,91.0,24.0,NaN,NaN,NaN,NaN
2011-05-01,299.0,54.0,52.0,53.0,61.0,69.0,81.0,28.0,NaN,NaN,NaN,NaN,NaN
2011-06-01,251.0,42.0,38.0,66.0,57.0,84.0,23.0,NaN,NaN,NaN,NaN,NaN,NaN
2011-07-01,202.0,36.0,42.0,47.0,54.0,25.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2011-08-01,176.0,36.0,45.0,41.0,23.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
# Calculate retention rates
cohort_sizes = cohort_counts.iloc[:, 0]
retention = cohort_counts.divide(cohort_sizes, axis=0)

print("Retention Rates:")
(retention.round(3) * 100)

Retention Rates:


CohortIndex,1,2,3,4,5,6,7,8,9,10,11,12,13
CohortMonth,,,,,,,,,,,,,
2010-12-01,100.0,41.8,38.9,44.4,44.0,47.3,42.4,40.9,41.1,44.2,40.7,56.6,32.2
2011-01-01,100.0,26.7,32.6,27.7,36.6,33.5,29.5,29.4,36.0,35.8,41.1,16.7,NaN
2011-02-01,100.0,19.8,19.8,28.9,28.4,25.1,27.0,29.6,26.5,32.5,8.1,NaN,NaN
2011-03-01,100.0,15.4,25.8,21.5,22.1,17.2,28.2,23.9,29.0,9.3,NaN,NaN,NaN
2011-04-01,100.0,22.9,20.8,22.9,21.4,23.2,23.5,27.8,7.3,NaN,NaN,NaN,NaN
2011-05-01,100.0,18.1,17.4,17.7,20.4,23.1,27.1,9.4,NaN,NaN,NaN,NaN,NaN
2011-06-01,100.0,16.7,15.1,26.3,22.7,33.5,9.2,NaN,NaN,NaN,NaN,NaN,NaN
2011-07-01,100.0,17.8,20.8,23.3,26.7,12.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2011-08-01,100.0,20.5,25.6,23.3,13.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



### 3.3 Cohort Visualization

In [10]:
# Create interactive retention heatmap with Plotly
fig = go.Figure(data=go.Heatmap(
    z=retention.values * 100,
    x=[f'Month {i}' for i in retention.columns],
    y=[date.strftime('%Y-%m') for date in retention.index],
    colorscale='Greens',
    text=np.round(retention.values * 100, 1),
    texttemplate='%{text}%',
    textfont={"size": 10},
    colorbar=dict(title="Retention %"),
    hoverongaps=False
))

fig.update_layout(
    title='Customer Retention Cohort Analysis',
    xaxis_title='Cohort Index (Months Since First Purchase)',
    yaxis_title='Cohort Month',
    height=600,
    width=1000
)

fig.show()

In [11]:
# Average revenue per cohort over time
grouping = online.groupby(['CohortMonth', 'CohortIndex'])
cohort_revenue = grouping['TotalSum'].mean()
cohort_revenue = cohort_revenue.reset_index()

average_revenue = cohort_revenue.pivot(index='CohortMonth',
                                        columns='CohortIndex',
                                        values='TotalSum')

# Visualize average revenue
fig = go.Figure(data=go.Heatmap(
    z=average_revenue.values,
    x=[f'Month {i}' for i in average_revenue.columns],
    y=[date.strftime('%Y-%m') for date in average_revenue.index],
    colorscale='Blues',
    text=np.round(average_revenue.values, 2),
    texttemplate='$%{text}',
    textfont={"size": 9},
    colorbar=dict(title="Avg Revenue"),
))

fig.update_layout(
    title='Average Revenue per Transaction by Cohort',
    xaxis_title='Cohort Index (Months Since First Purchase)',
    yaxis_title='Cohort Month',
    height=600,
    width=1000
)

fig.show()

## 4. RFMT Metrics Calculation

### 4.1 Calculate Base RFM Metrics

In [12]:
# Calculate RFM metrics at customer level
datamart_rfm = online.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  # Recency
    'InvoiceNo': 'count',  # Frequency
    'TotalSum': 'sum'  # Monetary Value
})

# Rename columns
datamart_rfm.rename(columns={
    'InvoiceDate': 'Recency',
    'InvoiceNo': 'Frequency',
    'TotalSum': 'MonetaryValue'
}, inplace=True)

print("RFM Metrics Summary:")
print(datamart_rfm.describe())
print(f"\nTotal customers: {len(datamart_rfm)}")

RFM Metrics Summary:
       Recency  Frequency  MonetaryValue
count  4269.00    4269.00        4269.00
mean     88.04      90.07        2018.45
std      94.31     224.45        8793.36
min       1.00       1.00           3.75
25%      18.00      17.00         305.28
50%      50.00      41.00         668.14
75%     134.00      98.00        1638.47
max     365.00    7707.00      280206.02

Total customers: 4269


### 4.2 Add Tenure Metric (RFMT)

In [13]:
# Calculate Tenure (days since first purchase)
first_purchase = online.groupby('CustomerID')['InvoiceDate'].min().reset_index()
first_purchase.columns = ['CustomerID', 'FirstPurchaseDate']

# Merge with datamart
datamart_rfm = datamart_rfm.merge(
    first_purchase.set_index('CustomerID'), 
    left_index=True, 
    right_index=True
)

# Calculate tenure in days
datamart_rfm['Tenure'] = (snapshot_date - datamart_rfm['FirstPurchaseDate']).dt.days

# Drop the date column, keep only tenure
datamart_rfm = datamart_rfm.drop('FirstPurchaseDate', axis=1)

# Reorder columns to RFMT
datamart_rfmt = datamart_rfm[['Recency', 'Frequency', 'MonetaryValue', 'Tenure']]

print("RFMT Metrics Summary:")
print(datamart_rfmt.describe())
datamart_rfmt.head(10)

RFMT Metrics Summary:
       Recency  Frequency  MonetaryValue   Tenure
count  4269.00    4269.00        4269.00  4269.00
mean     88.04      90.07        2018.45   211.59
std      94.31     224.45        8793.36   112.91
min       1.00       1.00           3.75     1.00
25%      18.00      17.00         305.28   101.00
50%      50.00      41.00         668.14   240.00
75%     134.00      98.00        1638.47   311.00
max     365.00    7707.00      280206.02   365.00


,Recency,Frequency,MonetaryValue,Tenure
CustomerID,,,,
12346.0,326,1,77183.60,326
12347.0,2,151,3598.21,317
12348.0,75,31,1797.24,358
12349.0,19,73,1757.55,19
12350.0,310,17,334.40,310
12352.0,36,85,2506.04,297
12353.0,204,4,89.00,204
12354.0,232,58,1079.40,232
12355.0,214,13,459.40,214


### 4.3 Visualize RFMT Distributions

In [14]:
# Create distribution plots for each metric
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Recency Distribution', 'Frequency Distribution',
                   'Monetary Value Distribution', 'Tenure Distribution')
)

# Recency
fig.add_trace(
    go.Histogram(x=datamart_rfmt['Recency'], name='Recency', 
                 marker_color='lightblue', nbinsx=50),
    row=1, col=1
)

# Frequency
fig.add_trace(
    go.Histogram(x=datamart_rfmt['Frequency'], name='Frequency',
                 marker_color='lightgreen', nbinsx=50),
    row=1, col=2
)

# Monetary Value
fig.add_trace(
    go.Histogram(x=datamart_rfmt['MonetaryValue'], name='Monetary',
                 marker_color='lightsalmon', nbinsx=50),
    row=2, col=1
)

# Tenure
fig.add_trace(
    go.Histogram(x=datamart_rfmt['Tenure'], name='Tenure',
                 marker_color='plum', nbinsx=50),
    row=2, col=2
)

fig.update_layout(
    height=800,
    width=1200,
    title_text="RFMT Metrics Distributions (Raw Data)",
    showlegend=False
)

fig.show()

In [15]:
import plotly.graph_objects as go
import numpy as np

# Correlation analysis (Assuming 'datamart_rfmt' is already defined)
correlation_matrix = datamart_rfmt.corr()

fig = go.Figure(data=go.Heatmap(
    z=correlation_matrix.values,
    x=correlation_matrix.columns,
    y=correlation_matrix.columns,
    colorscale='RdBu',
    zmid=0,
    text=np.round(correlation_matrix.values, 2),
    # Add these two lines to display that text on the plot:
    texttemplate="%{text}",
    textfont={"size": 12}, 
    colorbar=dict(title="Correlation")
))

fig.update_layout(
    title='RFMT Metrics Correlation Matrix',
    height=600,
    width=700
)

fig.show()

print("\nKey Insights:")
print(f"• Frequency-Monetary correlation: {correlation_matrix.loc['Frequency', 'MonetaryValue']:.3f}")
print(f"• Tenure-Frequency correlation: {correlation_matrix.loc['Tenure', 'Frequency']:.3f}")
print(f"• Recency-Tenure correlation: {correlation_matrix.loc['Recency', 'Tenure']:.3f}")


Key Insights:
• Frequency-Monetary correlation: 0.425
• Tenure-Frequency correlation: 0.202
• Recency-Tenure correlation: 0.270


## 5. Traditional RFM Segmentation

### 5.1 Create RFM Quartiles

In [16]:
# Create a copy for traditional RFM analysis
datamart_rfm_simple = datamart_rfmt[['Recency', 'Frequency', 'MonetaryValue']].copy()

# Create quartile labels
r_labels = list(range(4, 0, -1))  # Reversed: lower recency is better
f_labels = list(range(1, 5))      # Higher frequency is better
m_labels = list(range(1, 5))      # Higher monetary is better

# Calculate quartiles
datamart_rfm_simple['R'] = pd.qcut(datamart_rfm_simple['Recency'], q=4, labels=r_labels, duplicates='drop')
datamart_rfm_simple['F'] = pd.qcut(datamart_rfm_simple['Frequency'], q=4, labels=f_labels, duplicates='drop')
datamart_rfm_simple['M'] = pd.qcut(datamart_rfm_simple['MonetaryValue'], q=4, labels=m_labels, duplicates='drop')

# Create RFM segment and score
datamart_rfm_simple['RFM_Segment'] = (
    datamart_rfm_simple['R'].astype(str) + 
    datamart_rfm_simple['F'].astype(str) + 
    datamart_rfm_simple['M'].astype(str)
)

datamart_rfm_simple['RFM_Score'] = (
    datamart_rfm_simple['R'].astype(int) + 
    datamart_rfm_simple['F'].astype(int) + 
    datamart_rfm_simple['M'].astype(int)
)

print("RFM Segmentation Complete!")
datamart_rfm_simple.head(10)

RFM Segmentation Complete!


,Recency,Frequency,MonetaryValue,R,F,M,RFM_Segment,RFM_Score
CustomerID,,,,,,,,
12346.0,326,1,77183.60,1,1,4,114,6
12347.0,2,151,3598.21,4,4,4,444,12
12348.0,75,31,1797.24,2,2,4,224,8
12349.0,19,73,1757.55,3,3,4,334,10
12350.0,310,17,334.40,1,1,2,112,4
12352.0,36,85,2506.04,3,3,4,334,10
12353.0,204,4,89.00,1,1,1,111,3
12354.0,232,58,1079.40,1,3,3,133,7
12355.0,214,13,459.40,1,1,2,112,4


### 5.2 Analyze RFM Segments

In [17]:
# Top 10 RFM segments by size
segment_counts = datamart_rfm_simple['RFM_Segment'].value_counts().head(10)

fig = go.Figure(data=[
    go.Bar(x=segment_counts.index, y=segment_counts.values,
           marker_color='steelblue',
           text=segment_counts.values,
           textposition='auto')
])

fig.update_layout(
    title='Top 10 RFM Segments by Customer Count',
    xaxis_title='RFM Segment',
    yaxis_title='Number of Customers',
    height=500
)

fig.show()

In [18]:
# RFM Score distribution
score_summary = datamart_rfm_simple.groupby('RFM_Score').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'MonetaryValue': ['mean', 'count']
}).round(1)

score_summary.columns = ['Avg_Recency', 'Avg_Frequency', 'Avg_Monetary', 'Count']
print("Summary by RFM Score:")
print(score_summary)

Summary by RFM Score:
           Avg_Recency  Avg_Frequency  Avg_Monetary  Count
RFM_Score                                                 
3                246.9            8.2         158.2    375
4                166.1           13.8         238.1    378
5                148.1           21.0         366.5    494
6                 88.8           27.6         803.6    459
7                 81.1           38.8         756.5    463
8                 59.5           54.8         988.4    454
9                 44.0           77.6        1808.8    394
10                31.4          111.5        2023.0    432
11                20.8          185.2        4107.6    380
12                 7.2          366.4        9025.0    440


### 5.3 Create Named Segments

In [19]:
# Segment customers into Gold, Silver, Bronze
def segment_customers(score):
    if score >= 9:
        return 'Gold'
    elif score >= 6:
        return 'Silver'
    else:
        return 'Bronze'

datamart_rfm_simple['Segment'] = datamart_rfm_simple['RFM_Score'].apply(segment_customers)

# Summary by segment
segment_summary = datamart_rfm_simple.groupby('Segment').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'MonetaryValue': ['mean', 'count']
}).round(1)

segment_summary.columns = ['Avg_Recency', 'Avg_Frequency', 'Avg_Monetary', 'Count']
print("\nSegment Summary:")
print(segment_summary)


Segment Summary:
         Avg_Recency  Avg_Frequency  Avg_Monetary  Count
Segment                                                 
Bronze         183.3           15.0         265.0   1247
Gold            25.5          188.5        4324.7   1646
Silver          76.5           40.3         848.7   1376


In [20]:
# Visualize segment distribution
segment_counts = datamart_rfm_simple['Segment'].value_counts()

fig = go.Figure(data=[
    go.Pie(labels=segment_counts.index, 
           values=segment_counts.values,
           hole=0.4,
           marker_colors=['#FFD700', '#C0C0C0', '#CD7F32'])
])

fig.update_layout(
    title='Customer Distribution by Segment',
    height=500,
    annotations=[dict(text='Customers', x=0.5, y=0.5, font_size=20, showarrow=False)]
)

fig.show()

In [21]:
# Calculate revenue contribution
revenue_by_segment = datamart_rfm_simple.groupby('Segment')['MonetaryValue'].sum().sort_values(ascending=False)
print("\nRevenue Contribution by Segment:")
for segment, revenue in revenue_by_segment.items():
    pct = (revenue / revenue_by_segment.sum()) * 100
    print(f"  {segment}: ${revenue:,.2f} ({pct:.1f}%)")


Revenue Contribution by Segment:
  Gold: $7,118,494.61 (82.6%)
  Silver: $1,167,825.27 (13.6%)
  Bronze: $330,422.97 (3.8%)


## 6. Data Preprocessing for K-Means Clustering

### 6.1 Handle Skewness

In [22]:
# Visualize distributions after log transformation
fig = make_subplots(
    rows=2, cols=4,
    subplot_titles=('Recency (Original)', 'Frequency (Original)', 
                   'Monetary (Original)', 'Tenure (Original)',
                   'Recency (Log)', 'Frequency (Log)',
                   'Monetary (Log)', 'Tenure (Log)')
)

metrics = ['Recency', 'Frequency', 'MonetaryValue', 'Tenure']
colors = ['lightblue', 'lightgreen', 'lightsalmon', 'plum']

# Original distributions
for i, (metric, color) in enumerate(zip(metrics, colors), 1):
    fig.add_trace(
        go.Histogram(x=datamart_rfmt[metric], name=metric, 
                    marker_color=color, nbinsx=50, showlegend=False),
        row=1, col=i
    )

# Log-transformed distributions
datamart_log = np.log(datamart_rfmt)

for i, (metric, color) in enumerate(zip(metrics, colors), 1):
    fig.add_trace(
        go.Histogram(x=datamart_log[metric], name=f'{metric} (log)',
                    marker_color=color, nbinsx=50, showlegend=False),
        row=2, col=i
    )

fig.update_layout(
    height=600,
    width=1400,
    title_text="RFMT Distributions: Original vs Log-Transformed"
)

fig.show()

### 6.2 Standardization

In [23]:
# Apply log transformation
datamart_log = np.log(datamart_rfmt)

# Standardize the data (mean=0, std=1)
scaler = StandardScaler()
scaler.fit(datamart_log)
datamart_normalized = scaler.transform(datamart_log)

# Convert back to DataFrame for easier handling
datamart_normalized_df = pd.DataFrame(
    datamart_normalized,
    index=datamart_rfmt.index,
    columns=datamart_rfmt.columns
)

print("Normalized Data Statistics:")
print(f"Mean: {datamart_normalized.mean(axis=0).round(3)}")
print(f"Std:  {datamart_normalized.std(axis=0).round(3)}")

print("\nFirst 5 rows of normalized data:")
datamart_normalized_df.head()


Normalized Data Statistics:
Mean: [-0. -0.  0.  0.]
Std:  [1. 1. 1. 1.]

First 5 rows of normalized data:


,Recency,Frequency,MonetaryValue,Tenure
CustomerID,,,,
12346.0,1.45,-2.79,3.72,0.77
12347.0,-2.14,1.02,1.28,0.74
12348.0,0.41,-0.18,0.73,0.88
12349.0,-0.56,0.47,0.71,-2.35
12350.0,1.41,-0.64,-0.61,0.72


## 7. K-Means Clustering

### 7.1 Elbow Method - Finding Optimal K


In [24]:
# Calculate SSE for different values of K
sse = {}
silhouette_scores = {}

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(datamart_normalized)
    sse[k] = kmeans.inertia_
    silhouette_scores[k] = silhouette_score(datamart_normalized, kmeans.labels_)

print("SSE and Silhouette Scores:")
for k in range(2, 11):
    print(f"k={k}: SSE={sse[k]:.2f}, Silhouette={silhouette_scores[k]:.3f}")

SSE and Silhouette Scores:
k=2: SSE=10713.81, Silhouette=0.335
k=3: SSE=7682.47, Silhouette=0.351
k=4: SSE=6295.84, Silhouette=0.298
k=5: SSE=5689.79, Silhouette=0.286
k=6: SSE=5094.77, Silhouette=0.252
k=7: SSE=4606.19, Silhouette=0.255
k=8: SSE=4266.32, Silhouette=0.258
k=9: SSE=3967.49, Silhouette=0.250
k=10: SSE=3735.43, Silhouette=0.244


In [25]:
# Create elbow plot with Plotly
fig = make_subplots(rows=1, cols=2, subplot_titles=('Elbow Method', 'Silhouette Score'))

# Elbow plot
fig.add_trace(
    go.Scatter(x=list(sse.keys()), y=list(sse.values()),
              mode='lines+markers', marker=dict(size=10, color='steelblue'),
              name='SSE'),
    row=1, col=1
)

# Silhouette score plot
fig.add_trace(
    go.Scatter(x=list(silhouette_scores.keys()), y=list(silhouette_scores.values()),
              mode='lines+markers', marker=dict(size=10, color='coral'),
              name='Silhouette Score'),
    row=1, col=2
)

fig.update_xaxes(title_text="Number of Clusters (k)", row=1, col=1)
fig.update_yaxes(title_text="SSE", row=1, col=1)
fig.update_xaxes(title_text="Number of Clusters (k)", row=1, col=2)
fig.update_yaxes(title_text="Silhouette Score", row=1, col=2)

fig.update_layout(height=500, width=1200, title_text="Optimal Number of Clusters Analysis")
fig.show()

# Identify optimal k
optimal_k = max(silhouette_scores, key=silhouette_scores.get)
print(f"\n✓ Optimal number of clusters based on Silhouette Score: {optimal_k}")


✓ Optimal number of clusters based on Silhouette Score: 3


### 7.2 Fit K-Means with Multiple Solutions

In [26]:
# We'll test k=3 and k=4 to compare
cluster_solutions = {}

for k in [3, 4]:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(datamart_normalized)
    
    # Store results
    cluster_solutions[k] = {
        'model': kmeans,
        'labels': kmeans.labels_,
        'inertia': kmeans.inertia_,
        'silhouette': silhouette_score(datamart_normalized, kmeans.labels_)
    }
    
    print(f"\nk={k} Results:")
    print(f"  Inertia: {kmeans.inertia_:.2f}")
    print(f"  Silhouette Score: {silhouette_score(datamart_normalized, kmeans.labels_):.3f}")
    print(f"  Cluster sizes: {np.bincount(kmeans.labels_)}")


k=3 Results:
  Inertia: 7682.47
  Silhouette Score: 0.351
  Cluster sizes: [ 948 1687 1634]

k=4 Results:
  Inertia: 6295.84
  Silhouette Score: 0.298
  Cluster sizes: [ 877 1420  856 1116]


In [27]:
import pickle

# Save to file 
with open('kmeans_model.pkl', 'wb') as f:
    pickle.dump(cluster_solutions[3]['model'], f)

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

## 8. Cluster Analysis and Profiling

### 8.1 Analyze K=3 Solution


In [28]:
# Add cluster labels to original RFMT data
datamart_rfmt_k3 = datamart_rfmt.copy()
datamart_rfmt_k3['Cluster'] = cluster_solutions[3]['labels']

# Calculate average RFMT values per cluster
cluster_profile_k3 = datamart_rfmt_k3.groupby('Cluster').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'MonetaryValue': 'mean',
    'Tenure': 'mean'
}).round(2)

cluster_profile_k3['Size'] = datamart_rfmt_k3.groupby('Cluster').size()
cluster_profile_k3['Size_Pct'] = (cluster_profile_k3['Size'] / len(datamart_rfmt_k3) * 100).round(1)

print("Cluster Profiles (k=3):")
print(cluster_profile_k3)

Cluster Profiles (k=3):
         Recency  Frequency  MonetaryValue  Tenure  Size  Size_Pct
Cluster                                                           
0          34.44      33.61         480.49   50.28   948      22.2
1         174.20      26.68         563.91  241.41  1687      39.5
2          30.20     188.29        4412.44  274.39  1634      38.3


In [29]:
# Visualize cluster profiles
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Average Recency', 'Average Frequency',
                   'Average Monetary Value', 'Average Tenure'),
    specs=[[{'type': 'bar'}, {'type': 'bar'}],
           [{'type': 'bar'}, {'type': 'bar'}]]
)

colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

# Recency
fig.add_trace(
    go.Bar(x=cluster_profile_k3.index, y=cluster_profile_k3['Recency'],
           marker_color=colors, text=cluster_profile_k3['Recency'].round(1),
           textposition='auto', name='Recency'),
    row=1, col=1
)

# Frequency
fig.add_trace(
    go.Bar(x=cluster_profile_k3.index, y=cluster_profile_k3['Frequency'],
           marker_color=colors, text=cluster_profile_k3['Frequency'].round(1),
           textposition='auto', name='Frequency'),
    row=1, col=2
)

# Monetary Value
fig.add_trace(
    go.Bar(x=cluster_profile_k3.index, y=cluster_profile_k3['MonetaryValue'],
           marker_color=colors, text=cluster_profile_k3['MonetaryValue'].round(1),
           textposition='auto', name='Monetary'),
    row=2, col=1
)

# Tenure
fig.add_trace(
    go.Bar(x=cluster_profile_k3.index, y=cluster_profile_k3['Tenure'],
           marker_color=colors, text=cluster_profile_k3['Tenure'].round(1),
           textposition='auto', name='Tenure'),
    row=2, col=2
)

fig.update_layout(
    height=800,
    width=1200,
    title_text="Cluster Profiles: Average RFMT Metrics (k=3)",
    showlegend=False
)

fig.update_xaxes(title_text="Cluster", row=1, col=1)
fig.update_xaxes(title_text="Cluster", row=1, col=2)
fig.update_xaxes(title_text="Cluster", row=2, col=1)
fig.update_xaxes(title_text="Cluster", row=2, col=2)

fig.show()

### 8.2 Snake Plot for Cluster Comparison

In [30]:
# Prepare data for snake plot
datamart_normalized_k3 = datamart_normalized_df.copy()
datamart_normalized_k3['Cluster'] = cluster_solutions[3]['labels']

# Calculate mean normalized values per cluster
cluster_avg_normalized = datamart_normalized_k3.groupby('Cluster').mean()

# Create snake plot
fig = go.Figure()

for cluster in cluster_avg_normalized.index:
    fig.add_trace(go.Scatter(
        x=cluster_avg_normalized.columns,
        y=cluster_avg_normalized.loc[cluster],
        mode='lines+markers',
        name=f'Cluster {cluster}',
        line=dict(width=3),
        marker=dict(size=10)
    ))

fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)

fig.update_layout(
    title='Snake Plot: Standardized RFMT Values by Cluster',
    xaxis_title='RFMT Metrics',
    yaxis_title='Standardized Values',
    height=600,
    width=1000,
    hovermode='x unified'
)

fig.show()

print("\nInterpretation Guide:")
print("• Values above 0: Above average for that metric")
print("• Values below 0: Below average for that metric")
print("• The further from 0, the more extreme the characteristic")


Interpretation Guide:
• Values above 0: Above average for that metric
• Values below 0: Below average for that metric
• The further from 0, the more extreme the characteristic


### 8.3 Relative Importance Heatmap

In [31]:
# Calculate relative importance
cluster_avg = datamart_rfmt_k3.groupby('Cluster')[['Recency', 'Frequency', 'MonetaryValue', 'Tenure']].mean()
population_avg = datamart_rfmt[['Recency', 'Frequency', 'MonetaryValue', 'Tenure']].mean()

relative_importance = (cluster_avg / population_avg - 1).round(2)

print("Relative Importance vs Population Average:")
print(relative_importance)

Relative Importance vs Population Average:
         Recency  Frequency  MonetaryValue  Tenure
Cluster                                           
0          -0.61      -0.63          -0.76   -0.76
1           0.98      -0.70          -0.72    0.14
2          -0.66       1.09           1.19    0.30


In [32]:
# Visualize relative importance
fig = go.Figure(data=go.Heatmap(
    z=relative_importance.values,
    x=relative_importance.columns,
    y=[f'Cluster {i}' for i in relative_importance.index],
    colorscale='RdYlGn',
    zmid=0,
    text=relative_importance.values,
    texttemplate='%{text}',
    textfont={"size": 14},
    colorbar=dict(title="Relative<br>Importance")
))

fig.update_layout(
    title='Relative Importance of RFMT Attributes by Cluster<br>(Compared to Population Average)',
    xaxis_title='RFMT Metrics',
    yaxis_title='',
    height=400,
    width=800
)

fig.show()

print("\nInterpretation:")
print("• Green (positive): Cluster is above population average")
print("• Red (negative): Cluster is below population average")
print("• Values close to 0: Similar to population average")


Interpretation:
• Green (positive): Cluster is above population average
• Red (negative): Cluster is below population average
• Values close to 0: Similar to population average


## 9. Cluster Interpretation & Business Insights

### 9.1 Name and Characterize Clusters

In [33]:
# Based on the analysis, assign meaningful names to clusters
cluster_names = {
    0: 'Champions',        # High frequency, high monetary, low recency, high tenure
    1: 'At Risk',          # High recency (inactive), lower frequency
    2: 'Promising'         # Medium values across metrics
}

# This mapping should be adjusted based on actual cluster characteristics
datamart_rfmt_k3['Segment_Name'] = datamart_rfmt_k3['Cluster'].map(cluster_names)

# Display cluster characteristics
print("Cluster Characterization:")
print("="*80)
for cluster_id, name in cluster_names.items():
    cluster_data = datamart_rfmt_k3[datamart_rfmt_k3['Cluster'] == cluster_id]
    print(f"\n{name} (Cluster {cluster_id}):")
    print(f"  Size: {len(cluster_data)} customers ({len(cluster_data)/len(datamart_rfmt_k3)*100:.1f}%)")
    print(f"  Avg Recency: {cluster_data['Recency'].mean():.1f} days")
    print(f"  Avg Frequency: {cluster_data['Frequency'].mean():.1f} orders")
    print(f"  Avg Monetary: ${cluster_data['MonetaryValue'].mean():.2f}")
    print(f"  Avg Tenure: {cluster_data['Tenure'].mean():.1f} days")
    print(f"  Total Revenue: ${cluster_data['MonetaryValue'].sum():,.2f}")

Cluster Characterization:

Champions (Cluster 0):
  Size: 948 customers (22.2%)
  Avg Recency: 34.4 days
  Avg Frequency: 33.6 orders
  Avg Monetary: $480.49
  Avg Tenure: 50.3 days
  Total Revenue: $455,505.41

At Risk (Cluster 1):
  Size: 1687 customers (39.5%)
  Avg Recency: 174.2 days
  Avg Frequency: 26.7 orders
  Avg Monetary: $563.91
  Avg Tenure: 241.4 days
  Total Revenue: $951,309.40

Promising (Cluster 2):
  Size: 1634 customers (38.3%)
  Avg Recency: 30.2 days
  Avg Frequency: 188.3 orders
  Avg Monetary: $4412.44
  Avg Tenure: 274.4 days
  Total Revenue: $7,209,928.04


### 9.2 Customer Distribution Visualization


In [34]:
# Create sunburst chart for hierarchical view
segment_counts = datamart_rfmt_k3['Segment_Name'].value_counts()

fig = go.Figure(go.Sunburst(
    labels=['All Customers'] + list(segment_counts.index),
    parents=[''] + ['All Customers'] * len(segment_counts),
    values=[segment_counts.sum()] + list(segment_counts.values),
    branchvalues="total",
    marker=dict(colors=['lightgray', '#FF6B6B', '#4ECDC4', '#45B7D1'])
))

fig.update_layout(
    title='Customer Segment Distribution',
    height=600,
    width=800
)

fig.show()

In [35]:
# Revenue contribution by segment
revenue_by_cluster = datamart_rfmt_k3.groupby('Segment_Name').agg({
    'MonetaryValue': 'sum',
    'Cluster': 'count'
}).rename(columns={'Cluster': 'Customer_Count'})

revenue_by_cluster['Avg_Revenue_Per_Customer'] = (
    revenue_by_cluster['MonetaryValue'] / revenue_by_cluster['Customer_Count']
)

revenue_by_cluster['Revenue_Share'] = (
    revenue_by_cluster['MonetaryValue'] / revenue_by_cluster['MonetaryValue'].sum() * 100
)

print("\nRevenue Analysis by Segment:")
print(revenue_by_cluster.round(2))


Revenue Analysis by Segment:
              MonetaryValue  Customer_Count  Avg_Revenue_Per_Customer  \
Segment_Name                                                            
At Risk            9.51e+05            1687                    563.91   
Champions          4.56e+05             948                    480.49   
Promising          7.21e+06            1634                   4412.44   

              Revenue_Share  
Segment_Name                 
At Risk               11.04  
Champions              5.29  
Promising             83.67  


In [36]:
# Visualize revenue contribution
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'bar'}, {'type': 'pie'}]],
    subplot_titles=('Total Revenue by Segment', 'Revenue Share %')
)

# Bar chart
fig.add_trace(
    go.Bar(x=revenue_by_cluster.index, 
           y=revenue_by_cluster['MonetaryValue'],
           marker_color=['#FF6B6B', '#4ECDC4', '#45B7D1'],
           text=[f'${v:,.0f}' for v in revenue_by_cluster['MonetaryValue']],
           textposition='auto'),
    row=1, col=1
)

# Pie chart
fig.add_trace(
    go.Pie(labels=revenue_by_cluster.index,
           values=revenue_by_cluster['MonetaryValue'],
           marker_colors=['#FF6B6B', '#4ECDC4', '#45B7D1'],
           textinfo='label+percent',
           hole=0.3),
    row=1, col=2
)

fig.update_layout(height=500, width=1200, showlegend=False)
fig.show()

### 9.3 3D Cluster Visualization

In [37]:
# Create 3D scatter plot of clusters (using first 3 principal metrics)
fig = px.scatter_3d(
    datamart_rfmt_k3,
    x='Recency',
    y='Frequency',
    z='MonetaryValue',
    color='Segment_Name',
    size='Tenure',
    hover_data=['Recency', 'Frequency', 'MonetaryValue', 'Tenure'],
    title='3D Visualization of Customer Segments (RFM + Tenure as size)',
    color_discrete_map={
        'Champions': '#FF6B6B',
        'At Risk': '#4ECDC4',
        'Promising': '#45B7D1'
    },
    height=700
)

fig.update_traces(marker=dict(line=dict(width=0)))
fig.show()

## 10. K=4 Cluster Analysis (Comparison)

In [38]:
# Analyze k=4 solution for comparison
datamart_rfmt_k4 = datamart_rfmt.copy()
datamart_rfmt_k4['Cluster'] = cluster_solutions[4]['labels']

# Calculate average RFMT values per cluster
cluster_profile_k4 = datamart_rfmt_k4.groupby('Cluster').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'MonetaryValue': 'mean',
    'Tenure': 'mean'
}).round(2)

cluster_profile_k4['Size'] = datamart_rfmt_k4.groupby('Cluster').size()
cluster_profile_k4['Size_Pct'] = (cluster_profile_k4['Size'] / len(datamart_rfmt_k4) * 100).round(1)

print("Cluster Profiles (k=4):")
print(cluster_profile_k4)

Cluster Profiles (k=4):
         Recency  Frequency  MonetaryValue  Tenure  Size  Size_Pct
Cluster                                                           
0          32.59      34.39         475.95   47.18   877      20.5
1          79.15      75.13        1439.50  251.85  1420      33.3
2          11.59     269.45        6795.32  286.98   856      20.1
3         201.58      15.27         303.28  231.75  1116      26.1


In [39]:
# Snake plot for k=4
datamart_normalized_k4 = datamart_normalized_df.copy()
datamart_normalized_k4['Cluster'] = cluster_solutions[4]['labels']

cluster_avg_normalized_k4 = datamart_normalized_k4.groupby('Cluster').mean()

fig = go.Figure()

for cluster in cluster_avg_normalized_k4.index:
    fig.add_trace(go.Scatter(
        x=cluster_avg_normalized_k4.columns,
        y=cluster_avg_normalized_k4.loc[cluster],
        mode='lines+markers',
        name=f'Cluster {cluster}',
        line=dict(width=3),
        marker=dict(size=10)
    ))

fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)

fig.update_layout(
    title='Snake Plot: Standardized RFMT Values by Cluster (k=4)',
    xaxis_title='RFMT Metrics',
    yaxis_title='Standardized Values',
    height=600,
    width=1000,
    hovermode='x unified'
)

fig.show()

In [40]:
# Compare k=3 vs k=4 using silhouette scores
print("\n" + "="*80)
print("CLUSTER SOLUTION COMPARISON")
print("="*80)
print(f"\nk=3:")
print(f"  Silhouette Score: {cluster_solutions[3]['silhouette']:.4f}")
print(f"  Inertia: {cluster_solutions[3]['inertia']:.2f}")
print(f"  Cluster Sizes: {np.bincount(cluster_solutions[3]['labels'])}")

print(f"\nk=4:")
print(f"  Silhouette Score: {cluster_solutions[4]['silhouette']:.4f}")
print(f"  Inertia: {cluster_solutions[4]['inertia']:.2f}")
print(f"  Cluster Sizes: {np.bincount(cluster_solutions[4]['labels'])}")

print("\n✓ Recommended Solution: k=3 (better balance of interpretability and performance)")



CLUSTER SOLUTION COMPARISON

k=3:
  Silhouette Score: 0.3511
  Inertia: 7682.47
  Cluster Sizes: [ 948 1687 1634]

k=4:
  Silhouette Score: 0.2976
  Inertia: 6295.84
  Cluster Sizes: [ 877 1420  856 1116]

✓ Recommended Solution: k=3 (better balance of interpretability and performance)



## 11. Marketing Strategies & Recommendations

In [41]:
# Generate targeted marketing recommendations for each segment
marketing_strategies = {
    'Champions': {
        'Characteristics': 'High-value, frequent, recent, loyal customers',
        'Strategies': [
            '🎁 VIP rewards and exclusive offers',
            '🌟 Early access to new products',
            '💎 Loyalty program upgrades',
            '🎯 Referral incentives',
            '📧 Personalized premium communications'
        ],
        'Priority': 'HIGH - Retain and Reward',
        'Budget_Allocation': '40%'
    },
    'At Risk': {
        'Characteristics': 'Previously valuable but showing signs of churn',
        'Strategies': [
            '🔔 Re-engagement campaigns',
            '💰 Win-back discounts and special offers',
            '📞 Personal outreach from account managers',
            '🎯 Targeted ads on social media',
            '📨 "We miss you" email campaigns'
        ],
        'Priority': 'HIGH - Win Back',
        'Budget_Allocation': '35%'
    },
    'Promising': {
        'Characteristics': 'Moderate engagement with growth potential',
        'Strategies': [
            '📈 Upsell and cross-sell campaigns',
            '🎓 Product education content',
            '💡 Personalized recommendations',
            '🎉 Limited-time offers to increase frequency',
            '📱 Multi-channel engagement'
        ],
        'Priority': 'MEDIUM - Develop',
        'Budget_Allocation': '25%'
    }
}

print("="*80)
print("MARKETING STRATEGIES BY CUSTOMER SEGMENT")
print("="*80)

for segment, details in marketing_strategies.items():
    cluster_size = len(datamart_rfmt_k3[datamart_rfmt_k3['Segment_Name'] == segment])
    cluster_revenue = datamart_rfmt_k3[datamart_rfmt_k3['Segment_Name'] == segment]['MonetaryValue'].sum()
    
    print(f"\n{'─'*80}")
    print(f"📊 {segment.upper()}")
    print(f"{'─'*80}")
    print(f"Size: {cluster_size} customers | Revenue: ${cluster_revenue:,.2f}")
    print(f"Priority: {details['Priority']}")
    print(f"Budget Allocation: {details['Budget_Allocation']}")
    print(f"\n{details['Characteristics']}")
    print("\nRecommended Strategies:")
    for strategy in details['Strategies']:
        print(f"  {strategy}")

MARKETING STRATEGIES BY CUSTOMER SEGMENT

────────────────────────────────────────────────────────────────────────────────
📊 CHAMPIONS
────────────────────────────────────────────────────────────────────────────────
Size: 948 customers | Revenue: $455,505.41
Priority: HIGH - Retain and Reward
Budget Allocation: 40%

High-value, frequent, recent, loyal customers

Recommended Strategies:
  🎁 VIP rewards and exclusive offers
  🌟 Early access to new products
  💎 Loyalty program upgrades
  🎯 Referral incentives
  📧 Personalized premium communications

────────────────────────────────────────────────────────────────────────────────
📊 AT RISK
────────────────────────────────────────────────────────────────────────────────
Size: 1687 customers | Revenue: $951,309.40
Priority: HIGH - Win Back
Budget Allocation: 35%

Previously valuable but showing signs of churn

Recommended Strategies:
  🔔 Re-engagement campaigns
  💰 Win-back discounts and special offers
  📞 Personal outreach from account mana

## 12. Churn Risk Analysis

In [42]:
# Identify at-risk customers based on recency
datamart_rfmt_k3['Churn_Risk'] = pd.cut(
    datamart_rfmt_k3['Recency'],
    bins=[0, 30, 60, 90, float('inf')],
    labels=['Low', 'Medium', 'High', 'Critical']
)

# Churn risk by segment
churn_risk_analysis = pd.crosstab(
    datamart_rfmt_k3['Segment_Name'],
    datamart_rfmt_k3['Churn_Risk'],
    normalize='index'
) * 100

print("\nChurn Risk Distribution by Segment (%):")
print(churn_risk_analysis.round(1))


Churn Risk Distribution by Segment (%):
Churn_Risk     Low  Medium  High  Critical
Segment_Name                              
At Risk        4.0     7.8  11.6      76.6
Champions     50.4    31.5  18.0       0.0
Promising     67.4    19.4   7.8       5.4


In [43]:
# Visualize churn risk
fig = go.Figure()

for risk_level in ['Low', 'Medium', 'High', 'Critical']:
    fig.add_trace(go.Bar(
        name=risk_level,
        x=churn_risk_analysis.index,
        y=churn_risk_analysis[risk_level],
        text=churn_risk_analysis[risk_level].round(1),
        texttemplate='%{text}%',
        textposition='auto'
    ))

fig.update_layout(
    title='Churn Risk Distribution by Customer Segment',
    xaxis_title='Segment',
    yaxis_title='Percentage of Customers',
    barmode='stack',
    height=600,
    width=1000
)

fig.show()

## 13. Export Results

In [45]:
# Prepare final dataset with all insights
final_dataset = datamart_rfmt_k3[[
    'Recency', 'Frequency', 'MonetaryValue', 'Tenure',
    'Cluster', 'Segment_Name', 
     'Churn_Risk'
]].copy()

# Add customer ID back
final_dataset = final_dataset.reset_index()

print("Final Dataset Preview:")
print(final_dataset.head())

# Save to CSV
final_dataset.to_csv('customer_segments_analysis.csv', index=False)
print("\n✓ Results exported to 'customer_segments_analysis.csv'")

Final Dataset Preview:
   CustomerID  Recency  Frequency  MonetaryValue  Tenure  Cluster  \
0     12346.0      326          1       77183.60     326        1   
1     12347.0        2        151        3598.21     317        2   
2     12348.0       75         31        1797.24     358        1   
3     12349.0       19         73        1757.55      19        0   
4     12350.0      310         17         334.40     310        1   

  Segment_Name Churn_Risk  
0      At Risk   Critical  
1    Promising        Low  
2      At Risk       High  
3    Champions        Low  
4      At Risk   Critical  

✓ Results exported to 'customer_segments_analysis.csv'


In [48]:
# Create executive summary
print("\n" + "="*80)
print("EXECUTIVE SUMMARY")
print("="*80)

total_customers = len(final_dataset)
total_revenue = final_dataset['MonetaryValue'].sum()

print(f"\n📊 OVERVIEW")
print(f"  Total Customers Analyzed: {total_customers:,}")
print(f"  Total Revenue: ${total_revenue:,.2f}")
print(f"  Average Customer Value: ${total_revenue/total_customers:,.2f}")
print(f"  Analysis Period: {online['InvoiceDate'].min().date()} to {online['InvoiceDate'].max().date()}")

print(f"\n🎯 SEGMENTATION RESULTS")
for segment in final_dataset['Segment_Name'].unique():
    segment_data = final_dataset[final_dataset['Segment_Name'] == segment]
    segment_pct = len(segment_data) / total_customers * 100
    segment_revenue = segment_data['MonetaryValue'].sum()
    revenue_pct = segment_revenue / total_revenue * 100
    
    
    print(f"\n  {segment}:")
    print(f"    • Size: {len(segment_data):,} customers ({segment_pct:.1f}%)")
    print(f"    • Revenue: ${segment_revenue:,.2f} ({revenue_pct:.1f}%)")
    print(f"    • Avg Recency: {segment_data['Recency'].mean():.0f} days")
    print(f"    • Avg Frequency: {segment_data['Frequency'].mean():.1f} orders")

print(f"\n💡 KEY INSIGHTS")
print(f"  • Top performing segment contributes {revenue_by_cluster['Revenue_Share'].max():.1f}% of revenue")
print(f"  • {len(final_dataset[final_dataset['Churn_Risk'] == 'Critical'])} customers at critical churn risk")
print(f"  • Average customer tenure: {final_dataset['Tenure'].mean():.0f} days")


print("\n" + "="*80)


EXECUTIVE SUMMARY

📊 OVERVIEW
  Total Customers Analyzed: 4,269
  Total Revenue: $8,616,742.85
  Average Customer Value: $2,018.45
  Analysis Period: 2010-12-09 to 2011-12-09

🎯 SEGMENTATION RESULTS

  At Risk:
    • Size: 1,687 customers (39.5%)
    • Revenue: $951,309.40 (11.0%)
    • Avg Recency: 174 days
    • Avg Frequency: 26.7 orders

  Promising:
    • Size: 1,634 customers (38.3%)
    • Revenue: $7,209,928.04 (83.7%)
    • Avg Recency: 30 days
    • Avg Frequency: 188.3 orders

  Champions:
    • Size: 948 customers (22.2%)
    • Revenue: $455,505.41 (5.3%)
    • Avg Recency: 34 days
    • Avg Frequency: 33.6 orders

💡 KEY INSIGHTS
  • Top performing segment contributes 83.7% of revenue
  • 1380 customers at critical churn risk
  • Average customer tenure: 212 days



## 15. Conclusions & Next Steps

### Key Findings

**1. Customer Segmentation**
- Successfully identified 3 distinct customer segments using RFMT metrics and K-Means clustering
- Each segment exhibits unique behavioral patterns requiring tailored marketing approaches

**2. Revenue Concentration**
- Champions segment drives the majority of revenue despite representing a smaller percentage of customers
- Classic Pareto principle applies: focus on retaining and growing high-value segments

**3. Churn Risk**
- Significant portion of customers showing early warning signs of churn
- Proactive intervention strategies needed for at-risk segment

**4. Tenure Impact**
- Adding Tenure to traditional RFM provides deeper insights into customer lifecycle
- Longer-tenured customers show higher loyalty and lifetime value

### Recommended Actions

**Immediate (0-30 days):**
1. Launch win-back campaign for "At Risk" customers
2. Implement VIP rewards program for "Champions"
3. Set up automated monitoring for churn risk indicators

**Short-term (1-3 months):**
1. Develop segment-specific marketing content and offers
2. Create personalized customer journeys for each segment
3. Establish CLV-based customer acquisition targets

**Long-term (3-12 months):**
1. Build predictive models for churn and CLV
2. Implement real-time segmentation in CRM system
3. Conduct quarterly segment performance reviews
4. Test and optimize marketing strategies per segment

### Future Enhancements

- **Advanced Analytics**: Implement survival analysis for churn prediction
- **Real-time Segmentation**: Integrate with production systems for dynamic segmentation
- **Additional Features**: Include product preferences, channel behavior, seasonality
- **A/B Testing**: Test marketing strategies within segments to optimize ROI
- **Predictive Modeling**: Build ML models for next purchase prediction and recommendation engines
